# Detección temprana de anomalías en palma de aceite — pipeline completo
### Sentinel-2 (2025) · Módulo 1 índices · Módulo 2 segmentación CNN · Módulo 3 detección autosupervisada

```
X  = imagen Sentinel-2 en la fecha t (10 bandas de reflectancia + máscara de nubes)
│
├── MÓDULO 1 · Índices espectrales            {X}                    ──► NDVI, GNDVI, NDRE, NDMI
│
├── MÓDULO 2 · Segmentación (U-Net)            {X, índices}           ──► máscaras: palma(t), palma de referencia,
│        etiquetas débiles: GlobalOilPalm 2019 (o etiquetas manuales)        no-dosel persistente, evaluable(t)
│
└── MÓDULO 3 · Detección de anomalías          {X, índices, máscaras} ──► ANOMALÍA(t)
         One-Class SVM · Isolation Forest · LOF · SimCLR · BYOL · Autoencoder  →  consenso por votos
```

**Dataset:** `mass-oil-palm-dataset`: 200 ubicaciones (`crop_id`, palma *industrial* y de *pequeños
productores*), 7–9 fechas de 2025 cada una, una carpeta por ubicación-fecha con 12 bandas (`A_Bxx_(Raw).tiff`,
reflectancia 0–1), `cloud_mask.tif` (**1 = píxel limpio, 0 = nube**) y `metadata.json`.

**Decisiones de diseño que atraviesan todo el notebook**

| Decisión | Por qué |
|---|---|
| División train / val / test **por ubicación** (estratificada por clase) | Evaluar en lotes que ningún modelo vio. Dividir píxeles de una misma imagen filtra información entre entrenamiento y prueba |
| "Lo normal" = píxeles de palma de las ubicaciones de **entrenamiento**, en todas sus fechas | Supuesto no supervisado: la gran mayoría del dosel está sano |
| Los 6 detectores se calibran **igual**: umbral = percentil `1 − CONTAMINATION` de sus puntajes en las ubicaciones de **validación** | Así el 3% significa lo mismo para todos y los mapas son comparables |
| Evaluación sin etiquetas de campo: (a) U-Net vs GlobalOilPalm, (b) **anomalías sintéticas de estrés** inyectadas en ubicaciones de test (AUROC por detector), (c) acuerdo entre detectores y persistencia temporal | No hay verdad de campo; estas métricas son *proxies* honestos, no precisión real |

**Cómo correrlo en Kaggle:** *Settings → Accelerator: GPU T4* e *Internet: On*; agrega el dataset y corre las
celdas en orden **en el editor** (la descarga de GlobalOilPalm pide autenticar Google Earth Engine con un enlace).
Primera corrida completa: del orden de 1–2 h (la mayor parte es la inferencia sobre las ~1.600 escenas). Todo lo entrenado se guarda en `/kaggle/working/salidas` y, con
`CARGAR_SI_EXISTE = True`, las corridas siguientes lo reutilizan.

> ⚠️ Lo que este notebook entrega es **anomalía espectral** respecto a la palma "normal" del dataset, no un
> diagnóstico de enfermedad: confirmar la causa requiere verificación en campo.


## 0. Instalación, imports y configuración


In [1]:
!pip install -q rasterio earthengine-api

import os, re, io, json, glob, math, time, copy, random, shutil, zipfile, warnings, contextlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from rasterio.transform import array_bounds
from scipy import ndimage
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "SIN GPU (activa GPU T4)")


def autocast():
    """fp16 en GPU (tensor cores) solo para inferencia; en CPU no hace nada."""
    return torch.autocast("cuda", dtype=torch.float16) if DEVICE.type == "cuda" else contextlib.nullcontext()


def rng_para(etapa):
    """Generador reproducible e independiente por etapa: cargar un modelo ya entrenado en vez de
    reentrenarlo no cambia el muestreo de las etapas siguientes."""
    return np.random.RandomState((SEED * 1000 + sum(map(ord, etapa))) % (2 ** 32))


Dispositivo: cpu | SIN GPU (activa GPU T4)


In [2]:
# ============================================================
# CONFIGURACIÓN GENERAL -- lo único que normalmente hay que tocar
# ============================================================
# --- Rutas ---
DATA_ROOT = "/kaggle/input"                 # se busca recursivamente todo metadata.json aquí dentro
OUT_DIR = "/kaggle/working/salidas"
SALIDAS_PREVIAS = None     # carpeta 'salidas' de una corrida anterior guardada como dataset de Kaggle
                           # (p. ej. "/kaggle/input/mis-salidas-palma/salidas"): reutiliza modelos y GlobalOilPalm
ETIQUETAS_MANUALES_DIR = None   # opcional: GeoTIFFs '{crop_id}.tif' etiquetados a mano (1 palma, 0 no palma,
                                # 255 sin etiqueta). Si existen, reemplazan a GlobalOilPalm en esa ubicación
CARGAR_SI_EXISTE = True    # si ya hay redes entrenadas (aquí o en SALIDAS_PREVIAS) se cargan en vez de reentrenar
MODELOS_DIR = os.path.join(OUT_DIR, "modelos")
GOP_DIR = os.path.join(OUT_DIR, "globaloilpalm")
for d in (OUT_DIR, MODELOS_DIR, GOP_DIR):
    os.makedirs(d, exist_ok=True)

# --- Datos ---
BANDAS  = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]  # B01/B09 (aerosol, vapor) fuera
INDICES = ["NDVI", "GNDVI", "NDRE", "NDMI"]
CANALES = BANDAS + INDICES                  # 14 canales = {X, índices}
VALOR_PIXEL_LIMPIO = 1     # cloud_mask.tif: 1 = limpio, 0 = nube/inválido (coincide con local_cloud_cover_pct)
NUBE_MAX_ESCENA = 40.0     # % de nube local máximo para usar una fecha
MIN_FRAC_VALIDA = 0.05     # escenas con < 5% de píxeles válidos se descartan
N_UBICACIONES_MAX = None   # p. ej. 30 para una prueba rápida; None = todas
FRAC_VAL, FRAC_TEST = 0.15, 0.15

# --- Módulo 2: segmentación ---
GEE_PROJECT = "solid-century-509705-e8"
GEE_SECRETO_KAGGLE = "GEE_SERVICE_ACCOUNT"  # opcional: JSON de una cuenta de servicio guardado en Kaggle Secrets
PERMITIR_PSEUDO_ETIQUETAS = True            # si no hay GlobalOilPalm ni etiqueta manual: NDVI alto y estable (débil)
TILE_SEG, BATCH_SEG = 64, 32
EPOCAS_SEG, PASOS_EPOCA_SEG, LR_SEG, PACIENCIA_SEG = 40, 150, 1e-3, 8
UMBRAL_PALMA = 0.5
FRAC_FECHAS_PALMA = 0.34   # palma de referencia = la U-Net dice "palma" en >= 1/3 de las fechas válidas
NDVI_NO_DOSEL, FRAC_NO_DOSEL = 0.40, 0.80   # no-dosel persistente = NDVI < 0.40 en >= 80% de las fechas válidas
MIN_FECHAS_TEMPORAL = 3
BUFFER_NUBE_PX = 2         # se excluye un halo de 2 px alrededor de cada nube (bordes semitransparentes)

# --- Módulo 3: detección ---
CONTAMINATION = 0.03       # fracción de píxeles normales (validación) que cada detector marca como anómalos
PATCH = 9                  # ventana espacio-espectral de los modelos profundos (9 px = 90 m)
N_PARCHES_ENTRENAMIENTO = 60000
N_REFERENCIA = 20000       # banco de "lo normal" para detectores clásicos y SSD
N_REFERENCIA_OCSVM = 10000 # el OCSVM escala ~cuadrático al entrenar: se usa un subconjunto
N_CALIBRACION = 20000      # píxeles de validación para fijar umbrales
N_EVALUACION = 10000       # píxeles de test para la evaluación con anomalías sintéticas
CONSENSO_MIN = 4           # un píxel es anomalía de consenso si >= 4 de los 6 detectores lo marcan
LOTE_INFERENCIA = 8192
INFERENCIA_SOLO_TEST = False   # True = mapas solo para las ubicaciones de test (más rápido)
AREA_MIN_REGION_PX = 5

# --- Exportación ---
GUARDAR_INDICES_TIF = True
COMPRIMIR_SALIDAS = True   # empaqueta los GeoTIFF por módulo en .zip (Kaggle limita el número de archivos de salida)
print("Configuración cargada.")


Configuración cargada.


In [3]:
# ============================================================
# UTILIDADES
# ============================================================
def ruta_existente(subcarpeta, nombre):
    """Busca un archivo ya generado en esta corrida o en SALIDAS_PREVIAS."""
    for base in [OUT_DIR] + ([SALIDAS_PREVIAS] if SALIDAS_PREVIAS else []):
        ruta = os.path.join(base, subcarpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    return None


def escribir_tif(ruta, arr, u, dtype, nodata=None, nombres=None):
    """GeoTIFF alineado a la grilla de la ubicación u."""
    arr = arr[None] if arr.ndim == 2 else arr
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with rasterio.open(ruta, "w", driver="GTiff", height=u["H"], width=u["W"], count=arr.shape[0],
                       dtype=dtype, crs=u["crs"], transform=u["transform"], nodata=nodata,
                       compress="deflate") as dst:
        dst.write(arr.astype(dtype))
        for k, n in enumerate(nombres or [], start=1):
            dst.set_band_description(k, n)


def area_pixel_ha(transform, crs, H, W):
    """Área de un píxel en ha. El dataset viene en EPSG:4326 (grados), así que se convierte a metros
    con la latitud del centro (~10 m x 10 m en Santander)."""
    if crs is not None and crs.is_geographic:
        _, sur, _, norte = array_bounds(H, W, transform)
        lat = (sur + norte) / 2
        return abs(transform.a) * 111320 * math.cos(math.radians(lat)) * abs(transform.e) * 110574 / 1e4
    return abs(transform.a * transform.e) / 1e4


def rgb_escena(u, t):
    """Color verdadero (B04, B03, B02) con estiramiento 2-98% sobre píxeles válidos."""
    x = u["bandas"][t][[BANDAS.index("B04"), BANDAS.index("B03"), BANDAS.index("B02")]].astype(np.float32)
    lo, hi = np.percentile(x[:, u["valido"][t]], [2, 98], axis=1)
    return np.clip((x - lo[:, None, None]) / (hi - lo + 1e-6)[:, None, None], 0, 1).transpose(1, 2, 0)


## 0.1 Catálogo del dataset y división por ubicación

Se leen solo los `metadata.json` (rápido, sin abrir rasters). Se descartan fechas con más de `NUBE_MAX_ESCENA`%
de nube local y se divide **por ubicación** en train / val / test, estratificando por clase (industrial /
pequeño productor) para que ambas estén representadas en cada conjunto.


In [4]:
t0 = time.time()
filas = []
for ruta in glob.glob(os.path.join(DATA_ROOT, "**", "metadata.json"), recursive=True):
    try:
        with open(ruta) as f:
            m = json.load(f)
    except Exception:
        continue
    if "crop_id" not in m or "date" not in m:
        continue
    filas.append(dict(crop_id=m["crop_id"], clase=m.get("class_name") or "desconocida",
                      clase_valor=m.get("class_value"), fecha=m["date"],
                      nube=float(m.get("local_cloud_cover_pct") or 0.0), area_ha=m.get("area_ha"),
                      status=m.get("status", "ok"), carpeta=os.path.dirname(ruta)))
catalogo = pd.DataFrame(filas)
assert len(catalogo), f"No encontré metadata.json dentro de {DATA_ROOT}. ¿Agregaste el dataset al notebook?"
catalogo = (catalogo[catalogo.status == "ok"].sort_values("nube")
            .drop_duplicates(["crop_id", "fecha"]).sort_values(["crop_id", "fecha"]).reset_index(drop=True))
print(f"Catálogo: {len(catalogo)} escenas de {catalogo.crop_id.nunique()} ubicaciones ({time.time() - t0:.0f} s)")

uso = catalogo[catalogo.nube <= NUBE_MAX_ESCENA].copy()
print(f"Escenas con nube local <= {NUBE_MAX_ESCENA}%: {len(uso)} "
      f"(se descartan {len(catalogo) - len(uso)})")

ubicaciones = (uso.groupby("crop_id")
               .agg(clase=("clase", "first"), clase_valor=("clase_valor", "first"),
                    area_ha=("area_ha", "first"), n_fechas=("fecha", "count")).reset_index())
if N_UBICACIONES_MAX:
    partes = []
    for _, g in ubicaciones.groupby("clase"):
        n = max(1, round(N_UBICACIONES_MAX * len(g) / len(ubicaciones)))
        partes.append(g.sample(min(n, len(g)), random_state=SEED))
    ubicaciones = pd.concat(partes).reset_index(drop=True)
    print(f"MODO PRUEBA: {len(ubicaciones)} ubicaciones")

rng = rng_para("split")
split = {}
for _, g in ubicaciones.groupby("clase"):
    ids = sorted(g.crop_id)
    rng.shuffle(ids)
    n = len(ids)
    n_val = int(round(FRAC_VAL * n)) if n >= 3 else 0
    n_test = int(round(FRAC_TEST * n)) if n >= 3 else 0
    for k, c in enumerate(ids):
        split[c] = "val" if k < n_val else ("test" if k < n_val + n_test else "train")
ubicaciones["split"] = ubicaciones.crop_id.map(split)
display(pd.crosstab(ubicaciones.clase, ubicaciones.split, margins=True, margins_name="total"))


Catálogo: 1668 escenas de 200 ubicaciones (34 s)
Escenas con nube local <= 40.0%: 1362 (se descartan 306)


split,test,train,val,total
clase,,,,
industrial,16,78,16,110
smallholder,14,62,14,90
total,30,140,30,200


# MÓDULO 1 — Índices espectrales
**Entrada:** X (10 bandas de reflectancia de la fecha t). **Salida:** NDVI, GNDVI, NDRE, NDMI.

| Índice | Fórmula | Qué capta en palma |
|---|---|---|
| NDVI | (B08 − B04) / (B08 + B04) | vigor y biomasa (se satura con dosel denso) |
| GNDVI | (B08 − B03) / (B08 + B03) | clorofila; más sensible que el NDVI en dosel cerrado |
| NDRE | (B08 − B05) / (B08 + B05) | borde rojo: estrés **temprano**, antes de que el NDVI caiga |
| NDMI | (B08 − B11) / (B08 + B11) | contenido de agua del dosel (estrés hídrico) |

Detalles de la lectura: las bandas ya vienen en reflectancia 0–1 y en una grilla común de ~10 m; si alguna fecha
trae un tamaño distinto se remuestrea a la grilla de la primera. Los píxeles inválidos (nube o sin dato) se
rellenan con la mediana de la escena **solo para que no contaminen los parches vecinos**: nunca se evalúan, porque
la máscara `valido` los excluye de todo. Todo se guarda en memoria en `float16` (unos 3 GB para el dataset
completo).


In [ ]:
PATRON_BANDA = re.compile(r"_B(\d{2}|8A)_", re.I)


def _archivos_bandas(carpeta):
    archivos = {}
    for f in os.listdir(carpeta):
        m = PATRON_BANDA.search(f)
        if m and f.lower().endswith((".tif", ".tiff")):
            archivos["B" + m.group(1).upper()] = os.path.join(carpeta, f)
    return archivos


def leer_escena(carpeta, forma=None):
    """X de una fecha: bandas (10, H, W) float32, máscara de píxeles válidos, transform y CRS."""
    archivos = _archivos_bandas(carpeta)
    faltan = [b for b in BANDAS if b not in archivos]
    if faltan:
        raise FileNotFoundError(f"faltan bandas {faltan}")
    with rasterio.open(archivos["B04"]) as ref:
        transform, crs = ref.transform, ref.crs
        if forma is None:
            forma = (ref.height, ref.width)
    capas = []
    for b in BANDAS:
        with rasterio.open(archivos[b]) as src:
            a = src.read(1, out_shape=forma, resampling=Resampling.bilinear).astype(np.float32)
            if src.nodata is not None and np.isfinite(src.nodata):
                a[a == src.nodata] = np.nan
        capas.append(a)
    bandas = np.stack(capas)
    if np.nanmax(bandas) > 2:            # por si alguna escena viniera en DN (x10000)
        bandas /= 10000.0
    ruta_nube = os.path.join(carpeta, "cloud_mask.tif")
    if os.path.exists(ruta_nube):
        with rasterio.open(ruta_nube) as src:
            limpio = src.read(1, out_shape=forma, resampling=Resampling.nearest) == VALOR_PIXEL_LIMPIO
    else:
        limpio = np.ones(forma, bool)
    valido = (limpio & np.isfinite(bandas).all(axis=0)
              & ((bandas[BANDAS.index("B08")] + bandas[BANDAS.index("B04")]) > 0))
    return bandas, valido, transform, crs


def _nd(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        r = (a - b) / (a + b)
    return np.clip(np.where(np.isfinite(r), r, 0.0), -1, 1)


def calcular_indices(bandas):
    """MÓDULO 1: {X} -> NDVI, GNDVI, NDRE, NDMI  (4, H, W)."""
    b = {n: bandas[k] for k, n in enumerate(BANDAS)}
    return np.stack([_nd(b["B08"], b["B04"]),    # NDVI
                     _nd(b["B08"], b["B03"]),    # GNDVI
                     _nd(b["B08"], b["B05"]),    # NDRE
                     _nd(b["B08"], b["B11"])]    # NDMI
                    ).astype(np.float32)


INFO_UBIC = ubicaciones.set_index("crop_id").to_dict("index")


def cargar_ubicacion(crop_id):
    """Serie temporal completa de una ubicación: bandas, índices y validez por fecha."""
    info = INFO_UBIC[crop_id]
    forma = transform = crs = None
    fechas, nubes, B, I, V = [], [], [], [], []
    for _, f in uso[uso.crop_id == crop_id].sort_values("fecha").iterrows():
        try:
            bandas, valido, tr, cr = leer_escena(f.carpeta, forma)
        except Exception as e:
            print(f"  {crop_id} {f.fecha}: se omite ({e})")
            continue
        if forma is None:
            forma, transform, crs = bandas.shape[1:], tr, cr
        if valido.mean() < MIN_FRAC_VALIDA:
            continue
        mediana = np.nanmedian(np.where(valido, bandas, np.nan).reshape(len(BANDAS), -1), axis=1)
        bandas[:, ~valido] = mediana[:, None]
        fechas.append(f.fecha)
        nubes.append(f.nube)
        B.append(bandas.astype(np.float16))
        I.append(calcular_indices(bandas).astype(np.float16))
        V.append(valido)
    if not fechas:
        return None
    H, W = forma
    return dict(crop_id=crop_id, clase=info["clase"], clase_valor=info["clase_valor"], split=info["split"],
                fechas=fechas, nube=nubes, bandas=np.stack(B), indices=np.stack(I), valido=np.stack(V),
                transform=transform, crs=crs, H=H, W=W, px_ha=area_pixel_ha(transform, crs, H, W))


In [ ]:
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    cargadas = list(ex.map(cargar_ubicacion, ubicaciones.crop_id))
UBIC = {u["crop_id"]: u for u in cargadas if u is not None}
ubicaciones = ubicaciones[ubicaciones.crop_id.isin(UBIC)].reset_index(drop=True)
IDS = {s: sorted(c for c, u in UBIC.items() if u["split"] == s) for s in ("train", "val", "test")}
assert IDS["train"] and IDS["val"] and IDS["test"], "Faltan ubicaciones en algún split (sube N_UBICACIONES_MAX)."

n_escenas = sum(len(u["fechas"]) for u in UBIC.values())
n_px = sum(u["valido"].size for u in UBIC.values())
gb = sum(u["bandas"].nbytes + u["indices"].nbytes + u["valido"].nbytes for u in UBIC.values()) / 1e9
print(f"Módulo 1 listo: {len(UBIC)} ubicaciones, {n_escenas} escenas, {n_px / 1e6:.1f} M píxeles-fecha "
      f"({gb:.1f} GB en memoria) en {time.time() - t0:.0f} s")
print({s: len(v) for s, v in IDS.items()})

# --- Normalización: media / desviación por canal sobre píxeles válidos de las ubicaciones de TRAIN ---
rng = rng_para("normalizacion")
muestras = []
for c in IDS["train"]:
    u = UBIC[c]
    for t in range(len(u["fechas"])):
        idx = np.flatnonzero(u["valido"][t])
        sel = rng.choice(idx, min(len(idx), 2000), replace=False)
        x = np.concatenate([u["bandas"][t].reshape(len(BANDAS), -1)[:, sel],
                            u["indices"][t].reshape(len(INDICES), -1)[:, sel]])
        muestras.append(x.T.astype(np.float32))
muestras = np.concatenate(muestras)
MU, SD = muestras.mean(0), muestras.std(0) + 1e-6
MU_T = torch.tensor(MU, device=DEVICE).view(1, -1, 1, 1)
SD_T = torch.tensor(SD, device=DEVICE).view(1, -1, 1, 1)
display(pd.DataFrame({"media": MU, "desv": SD}, index=CANALES).T.round(4))


def cubo_norm(u, t, filas=slice(None), cols=slice(None)):
    """{X, índices} normalizado (14, h, w) de la fecha t (opcionalmente un recorte)."""
    x = np.concatenate([u["bandas"][t][:, filas, cols], u["indices"][t][:, filas, cols]]).astype(np.float32)
    return (x - MU[:, None, None]) / SD[:, None, None]


In [ ]:
# --- Vista de control del Módulo 1 + exportación de índices (GeoTIFF int16, índice x 10000) ---
c_ej = max(IDS["test"], key=lambda c: UBIC[c]["valido"].sum())
u = UBIC[c_ej]
t = int(np.argmax(u["valido"].mean(axis=(1, 2))))
fig, ax = plt.subplots(1, 5, figsize=(25, 5.5))
ax[0].imshow(rgb_escena(u, t)); ax[0].set_title(f"{c_ej} · {u['fechas'][t]} · RGB")
for k, nombre in enumerate(INDICES, start=1):
    im = ax[k].imshow(np.where(u["valido"][t], u["indices"][t][k - 1].astype(np.float32), np.nan),
                      cmap="RdYlGn", vmin=-0.2, vmax=1)
    ax[k].set_title(nombre)
for a in ax:
    a.axis("off")
plt.colorbar(im, ax=ax[1:], fraction=0.015)
plt.show()

if GUARDAR_INDICES_TIF:
    t0 = time.time()
    for c, u in UBIC.items():
        for t, fecha in enumerate(u["fechas"]):
            arr = np.round(u["indices"][t].astype(np.float32) * 10000).astype(np.int16)
            arr[:, ~u["valido"][t]] = -32768
            escribir_tif(os.path.join(OUT_DIR, "modulo1_indices", c, f"{fecha}_indices.tif"),
                         arr, u, "int16", -32768, INDICES)
    print(f"Índices exportados en {OUT_DIR}/modulo1_indices ({time.time() - t0:.0f} s)")


# MÓDULO 2 — Segmentación de palma con CNN (U-Net)
**Entrada:** {X, índices} (14 canales). **Salida:** máscaras.

**Etiquetas para entrenar la CNN.** El dataset no trae polígonos de palma, así que se usa **GlobalOilPalm**
(Descals et al.; mapa global de palma a 10 m, clases 1 = industrial, 2 = pequeño productor, 3 = otro), descargado
de Google Earth Engine para el recuadro de cada ubicación y alineado píxel a píxel a su grilla. Es de **2019**, por
lo que se trata como **etiqueta débil**: la U-Net aprende la apariencia espectral de la palma a partir de ella y
produce un mapa **actualizado para cada fecha de 2025**. Si pones GeoTIFFs etiquetados a mano en
`ETIQUETAS_MANUALES_DIR`, esos reemplazan a GlobalOilPalm en sus ubicaciones (pseudoetiquetas + etiquetado manual).
Las nubes se ignoran en la pérdida.

**Autenticación de Earth Engine en Kaggle** (solo la primera vez; luego se usa la caché en `salidas/globaloilpalm`):
- **Opción A (interactiva):** corre la celda en el editor, abre el enlace, autoriza y pega el código.
- **Opción B (para "Save & Run All"):** guarda el JSON de una cuenta de servicio de GEE en *Add-ons → Secrets*
  con el nombre `GEE_SERVICE_ACCOUNT`.
- Tras la primera descarga puedes guardar `salidas/` como dataset y apuntar `SALIDAS_PREVIAS` a él.


In [ ]:
# ============================================================
# ETIQUETAS: etiquetas manuales > GlobalOilPalm (GEE, en caché) > pseudoetiquetas
# ============================================================
def iniciar_gee():
    import ee
    try:
        ee.Initialize(project=GEE_PROJECT)
        return True
    except Exception:
        pass
    try:                                           # Opción B: cuenta de servicio en Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        clave = UserSecretsClient().get_secret(GEE_SECRETO_KAGGLE)
        cred = ee.ServiceAccountCredentials(json.loads(clave)["client_email"], key_data=clave)
        ee.Initialize(cred, project=GEE_PROJECT)
        print("GEE: autenticado con cuenta de servicio.")
        return True
    except Exception:
        pass
    try:                                           # Opción A: enlace + código
        ee.Authenticate(auth_mode="notebook")
        ee.Initialize(project=GEE_PROJECT)
        print("GEE: autenticado.")
        return True
    except Exception as e:
        print("GEE no disponible:", e)
        return False


def descargar_gop(u):
    """GlobalOilPalm (clasificación 1/2/3) del recuadro de la ubicación, con ~200 m de margen."""
    import ee
    oeste, sur, este, norte = array_bounds(u["H"], u["W"], u["transform"])
    if u["crs"] is not None and not u["crs"].is_geographic:
        oeste, sur, este, norte = transform_bounds(u["crs"], "EPSG:4326", oeste, sur, este, norte)
    m = 0.002
    region = ee.Geometry.Rectangle([oeste - m, sur - m, este + m, norte + m], proj="EPSG:4326", geodesic=False)
    img = ee.ImageCollection("BIOPAMA/GlobalOilPalm/v1").select("classification").mosaic().unmask(3).toByte()
    url = img.getDownloadURL({"region": region, "scale": 10, "crs": "EPSG:4326", "format": "GEO_TIFF"})
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    contenido = r.content
    if contenido[:2] == b"PK":                     # por si GEE lo entrega comprimido
        with zipfile.ZipFile(io.BytesIO(contenido)) as z:
            contenido = z.read([n for n in z.namelist() if n.lower().endswith(".tif")][0])
    ruta = os.path.join(GOP_DIR, f"{u['crop_id']}.tif")
    with open(ruta, "wb") as f:
        f.write(contenido)
    return ruta


def alinear_a_grilla(ruta, u, relleno):
    """Reproyecta (vecino más cercano) un raster a la grilla exacta de la ubicación."""
    destino = np.full((u["H"], u["W"]), relleno, np.uint8)
    with rasterio.open(ruta) as src:
        reproject(source=rasterio.band(src, 1), destination=destino, src_transform=src.transform,
                  src_crs=src.crs, dst_transform=u["transform"], dst_crs=u["crs"],
                  resampling=Resampling.nearest)
    return destino


def pseudo_etiqueta(u):
    """Último recurso: NDVI mediano >= 0.75 en el año. Marca vegetación densa y estable, NO palma
    específicamente: úsalo solo si no hay GlobalOilPalm ni etiquetas manuales."""
    ndvi = np.where(u["valido"], u["indices"][:, 0].astype(np.float32), np.nan)
    med = np.nan_to_num(np.nanmedian(ndvi, axis=0), nan=0.0)
    clase = int(u["clase_valor"]) if u["clase_valor"] in (1, 2) else 1
    return np.where(med >= 0.75, clase, 3).astype(np.uint8)


def ruta_manual(crop_id):
    if not ETIQUETAS_MANUALES_DIR:
        return None
    ruta = os.path.join(ETIQUETAS_MANUALES_DIR, f"{crop_id}.tif")
    return ruta if os.path.exists(ruta) else None


t0 = time.time()
faltan_gop = [c for c in UBIC if ruta_manual(c) is None and ruta_existente("globaloilpalm", f"{c}.tif") is None]
GEE_OK = iniciar_gee() if faltan_gop else False
if faltan_gop:
    print(f"Descargando GlobalOilPalm para {len(faltan_gop)} ubicaciones...")
for k, (c, u) in enumerate(UBIC.items()):
    ruta_m = ruta_manual(c)
    if ruta_m:                                     # 1 palma, 0 no palma, 255 sin etiqueta
        manual = alinear_a_grilla(ruta_m, u, 255)
        u["gop"] = np.where(manual == 1, 1, np.where(manual == 0, 3, 255)).astype(np.uint8)
        u["fuente_etiqueta"] = "manual"
        continue
    ruta = ruta_existente("globaloilpalm", f"{c}.tif")
    if ruta is None and GEE_OK:
        try:
            ruta = descargar_gop(u)
        except Exception as e:
            print(f"  {c}: falló la descarga ({e})")
    if ruta:
        u["gop"] = alinear_a_grilla(ruta, u, 3)
        u["fuente_etiqueta"] = "GlobalOilPalm"
    else:
        assert PERMITIR_PSEUDO_ETIQUETAS, f"{c}: sin GlobalOilPalm ni etiqueta manual."
        u["gop"] = pseudo_etiqueta(u)
        u["fuente_etiqueta"] = "pseudo-NDVI"
    if (k + 1) % 25 == 0:
        print(f"  {k + 1}/{len(UBIC)} ({time.time() - t0:.0f} s)")

fuentes = pd.Series({c: u["fuente_etiqueta"] for c, u in UBIC.items()}).value_counts()
print("Fuente de etiquetas por ubicación:", fuentes.to_dict())
if "pseudo-NDVI" in fuentes:
    print("⚠️  Hay ubicaciones con pseudoetiquetas de NDVI: la 'palma' ahí es vegetación densa, no palma verificada.")

cobertura = pd.DataFrame([dict(crop_id=c, clase=u["clase"], split=u["split"],
                               pct_palma=100 * np.isin(u["gop"], (1, 2)).mean(),
                               pct_clase_coincide=100 * np.mean(u["gop"][np.isin(u["gop"], (1, 2))]
                                                                == u["clase_valor"])
                               if np.isin(u["gop"], (1, 2)).any() else np.nan)
                          for c, u in UBIC.items()])
print("\n% de palma según la etiqueta, y % de esa palma cuya clase coincide con la del metadata:")
display(cobertura.groupby("clase")[["pct_palma", "pct_clase_coincide"]].describe().round(1))


## 2.1 U-Net: arquitectura y entrenamiento

U-Net clásica de 3 niveles (32-64-128-256 canales, ~1.9 M de parámetros), entrenada con recortes de 64×64
tomados al azar de las ubicaciones de **train** (todas las fechas), con rotaciones y volteos. Pérdida =
**BCE + Dice** enmascaradas (sin nubes ni píxeles sin etiqueta). Se guarda el modelo con mejor **IoU en
validación** frente a la etiqueta (early stopping).


In [ ]:
class DobleConv(nn.Sequential):
    def __init__(self, c_in, c_out):
        super().__init__(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))


class UNet(nn.Module):
    def __init__(self, c_in=len(CANALES), base=32):
        super().__init__()
        self.e1, self.e2 = DobleConv(c_in, base), DobleConv(base, base * 2)
        self.e3, self.fondo = DobleConv(base * 2, base * 4), DobleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.s3, self.d3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2), DobleConv(base * 8, base * 4)
        self.s2, self.d2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2), DobleConv(base * 4, base * 2)
        self.s1, self.d1 = nn.ConvTranspose2d(base * 2, base, 2, 2), DobleConv(base * 2, base)
        self.salida = nn.Conv2d(base, 1, 1)

    def forward(self, x):                           # x: (N, 14, H, W), H y W múltiplos de 8
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        f = self.fondo(self.pool(e3))
        d3 = self.d3(torch.cat([self.s3(f), e3], 1))
        d2 = self.d2(torch.cat([self.s2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.s1(d2), e1], 1))
        return self.salida(d1)[:, 0]                # logits (N, H, W)


def muestrear_tiles(ids, n, rng):
    X = np.zeros((n, len(CANALES), TILE_SEG, TILE_SEG), np.float32)
    Y = np.zeros((n, TILE_SEG, TILE_SEG), np.float32)
    M = np.zeros((n, TILE_SEG, TILE_SEG), np.float32)
    for k in range(n):
        u = UBIC[ids[rng.randint(len(ids))]]
        t = rng.randint(len(u["fechas"]))
        h, w = min(TILE_SEG, u["H"]), min(TILE_SEG, u["W"])
        i, j = rng.randint(u["H"] - h + 1), rng.randint(u["W"] - w + 1)
        fs, cs = slice(i, i + h), slice(j, j + w)
        x = cubo_norm(u, t, fs, cs)
        y = np.isin(u["gop"][fs, cs], (1, 2))
        m = u["valido"][t][fs, cs] & (u["gop"][fs, cs] != 255)
        r = rng.randint(4)
        x, y, m = np.rot90(x, r, (1, 2)), np.rot90(y, r), np.rot90(m, r)
        if rng.rand() < 0.5:
            x, y, m = x[:, :, ::-1], y[:, ::-1], m[:, ::-1]
        hh, ww = y.shape
        X[k, :, :hh, :ww], Y[k, :hh, :ww], M[k, :hh, :ww] = x, y, m
    return tuple(torch.from_numpy(a).to(DEVICE) for a in (X, Y, M))


def perdida_segmentacion(logits, y, m):
    bce = (F.binary_cross_entropy_with_logits(logits, y, reduction="none") * m).sum() / m.sum().clamp_min(1)
    p = torch.sigmoid(logits) * m
    dice = 1 - (2 * (p * y).sum() + 1) / (p.sum() + (y * m).sum() + 1)
    return bce + dice


@torch.no_grad()
def prob_palma(modelo, u, t):
    """Probabilidad de palma de la escena completa (la U-Net es totalmente convolucional)."""
    modelo.eval()
    H, W = u["H"], u["W"]
    x = torch.from_numpy(cubo_norm(u, t))[None].to(DEVICE)
    x = F.pad(x, (0, (-W) % 8, 0, (-H) % 8), mode="reflect")
    with autocast():
        p = torch.sigmoid(modelo(x).float())
    return p[0, :H, :W].cpu().numpy()


def evaluar_segmentacion(modelo, ids):
    """Acuerdo con la etiqueta (GlobalOilPalm 2019 / manual) en píxeles válidos."""
    tp = fp = fn = tn = 0
    for c in ids:
        u = UBIC[c]
        ref, etiquetado = np.isin(u["gop"], (1, 2)), u["gop"] != 255
        for t in range(len(u["fechas"])):
            pred = prob_palma(modelo, u, t) >= UMBRAL_PALMA
            v = u["valido"][t] & etiquetado
            tp += int(np.sum(pred & ref & v)); fp += int(np.sum(pred & ~ref & v))
            fn += int(np.sum(~pred & ref & v)); tn += int(np.sum(~pred & ~ref & v))
    prec, rec = tp / max(1, tp + fp), tp / max(1, tp + fn)
    return dict(IoU=tp / max(1, tp + fp + fn), F1=2 * prec * rec / max(1e-9, prec + rec),
                precision=prec, recall=rec, exactitud=(tp + tn) / max(1, tp + fp + fn + tn))


def entrenar_unet():
    torch.manual_seed(SEED)
    modelo = UNet().to(DEVICE)
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR_SEG, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR_SEG, total_steps=EPOCAS_SEG * PASOS_EPOCA_SEG,
                                                pct_start=0.1)
    rng = rng_para("tiles")
    mejor, estado, sin_mejora, hist, t0 = -1.0, None, 0, [], time.time()
    for ep in range(EPOCAS_SEG):
        modelo.train()
        suma = 0.0
        for _ in range(PASOS_EPOCA_SEG):
            X, Y, M = muestrear_tiles(IDS["train"], BATCH_SEG, rng)
            perdida = perdida_segmentacion(modelo(X), Y, M)
            opt.zero_grad(set_to_none=True)
            perdida.backward()
            opt.step()
            sched.step()
            suma += perdida.item()
        met = evaluar_segmentacion(modelo, IDS["val"])
        hist.append(dict(epoca=ep + 1, perdida=suma / PASOS_EPOCA_SEG, **{f"val_{k}": v for k, v in met.items()}))
        print(f"  U-Net época {ep + 1:2d} | pérdida {hist[-1]['perdida']:.4f} | IoU val {met['IoU']:.3f} "
              f"| F1 val {met['F1']:.3f} | {time.time() - t0:.0f} s")
        if met["IoU"] > mejor:
            mejor, estado, sin_mejora = met["IoU"], copy.deepcopy(modelo.state_dict()), 0
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA_SEG:
                print("  Early stopping.")
                break
    modelo.load_state_dict(estado)
    return modelo.eval(), pd.DataFrame(hist)


ruta = ruta_existente("modelos", "unet_palma.pt")
if CARGAR_SI_EXISTE and ruta:
    unet = UNet().to(DEVICE)
    unet.load_state_dict(torch.load(ruta, map_location=DEVICE))
    unet.eval()
    print("U-Net cargada de", ruta)
else:
    print("Entrenando U-Net...")
    unet, hist_seg = entrenar_unet()
    torch.save(unet.state_dict(), os.path.join(MODELOS_DIR, "unet_palma.pt"))
    hist_seg.to_csv(os.path.join(OUT_DIR, "unet_historial.csv"), index=False)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(hist_seg.epoca, hist_seg.perdida); ax[0].set_title("Pérdida (BCE + Dice)")
    ax[1].plot(hist_seg.epoca, hist_seg.val_IoU, label="IoU"); ax[1].plot(hist_seg.epoca, hist_seg.val_F1, label="F1")
    ax[1].set_title("Validación vs etiqueta"); ax[1].legend()
    for a in ax:
        a.set_xlabel("época"); a.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "unet_curvas.png"), dpi=110); plt.show()

metricas_seg = pd.DataFrame({s: evaluar_segmentacion(unet, IDS[s]) for s in ("train", "val", "test")}).T
for clase in sorted({UBIC[c]["clase"] for c in IDS["test"]}):
    metricas_seg.loc[f"test · {clase}"] = pd.Series(
        evaluar_segmentacion(unet, [c for c in IDS["test"] if UBIC[c]["clase"] == clase]))
metricas_seg.to_csv(os.path.join(OUT_DIR, "segmentacion_metricas.csv"))
print("Acuerdo de la U-Net con la etiqueta (GlobalOilPalm 2019 / manual). No es exactitud contra verdad de campo:")
display(metricas_seg.round(3))


## 2.2 Máscaras (salida del Módulo 2)

| Máscara | Definición | Para qué |
|---|---|---|
| `palma(t)` | U-Net ≥ 0.5 en la fecha t (píxeles válidos) | segmentación por fecha, tal como pide el flujo |
| `palma de referencia` | la U-Net dice "palma" en **≥ 1/3** de las fechas válidas | región donde se buscan anomalías |
| `no-dosel persistente` | NDVI < 0.40 en ≥ 80% de las fechas válidas (con ≥ 3 fechas) | caminos internos, campamentos, suelo desnudo |
| `evaluable(t)` | palma de referencia ∧ válido(t) ∧ ¬no-dosel ∧ lejos de nubes (2 px) | píxeles que analiza el Módulo 3 |

**¿Por qué la región de búsqueda es la palma de *referencia* y no `palma(t)`?** Una palma que se enferma o se
seca cambia su firma espectral, y en esa fecha la U-Net puede dejar de reconocerla como palma. Si el Módulo 3 solo
mirara `palma(t)`, la anomalía se escondería justo cuando aparece (el mismo problema que tenía la regla
"NDVI < 0.45 = camino"). Con el criterio temporal basta que el píxel haya sido palma en una parte del año.

**Caminos.** La cuadrícula de caminos de las plantaciones industriales tiene NDVI bajo **siempre**; una palma que se
deteriora lo tiene **desde cierta fecha**. La regla de no-dosel persistente usa esa diferencia. Además se reporta
`palma 2019 sin dosel 2025`: píxeles que GlobalOilPalm marcaba como palma y que en todo 2025 están sin dosel
(renovación, tala o pérdida total; es un cambio que no llega a ser anomalía puntual).


In [ ]:
def generar_mascaras(u):
    T = len(u["fechas"])
    palma_t = np.stack([(prob_palma(unet, u, t) >= UMBRAL_PALMA) & u["valido"][t] for t in range(T)])
    n_valid = u["valido"].sum(0)
    palma_ref = (palma_t.sum(0) >= FRAC_FECHAS_PALMA * n_valid) & (n_valid > 0)
    ndvi = u["indices"][:, 0].astype(np.float32)
    bajo = ((ndvi < NDVI_NO_DOSEL) & u["valido"]).sum(0)
    no_dosel = (n_valid >= MIN_FECHAS_TEMPORAL) & (bajo >= FRAC_NO_DOSEL * n_valid)
    evaluable = np.zeros_like(palma_t)
    for t in range(T):
        cerca_nube = ~u["valido"][t]
        if BUFFER_NUBE_PX:                         # halo cuadrado de BUFFER_NUBE_PX píxeles
            cerca_nube = ndimage.binary_dilation(cerca_nube, structure=np.ones((3, 3), bool),
                                                 iterations=BUFFER_NUBE_PX)
        evaluable[t] = palma_ref & ~no_dosel & ~cerca_nube
    u.update(palma_t=palma_t, palma_ref=palma_ref, no_dosel=no_dosel, evaluable=evaluable,
             palma2019_sin_dosel=np.isin(u["gop"], (1, 2)) & no_dosel)


t0 = time.time()
for k, u in enumerate(UBIC.values()):
    generar_mascaras(u)
    if (k + 1) % 50 == 0:
        print(f"  máscaras {k + 1}/{len(UBIC)} ({time.time() - t0:.0f} s)")

resumen_m2 = pd.DataFrame([dict(crop_id=c, clase=u["clase"], split=u["split"],
                                pct_palma_ref=100 * u["palma_ref"].mean(),
                                pct_no_dosel_en_palma=100 * (u["no_dosel"] & u["palma_ref"]).sum()
                                / max(1, u["palma_ref"].sum()),
                                ha_palma2019_sin_dosel=u["palma2019_sin_dosel"].sum() * u["px_ha"],
                                px_evaluables=int(u["evaluable"].sum()))
                           for c, u in UBIC.items()])
resumen_m2.to_csv(os.path.join(OUT_DIR, "modulo2_resumen.csv"), index=False)
print(f"Módulo 2 listo en {time.time() - t0:.0f} s. "
      f"Píxeles-fecha evaluables en total: {resumen_m2.px_evaluables.sum() / 1e6:.1f} M")
display(resumen_m2.groupby("clase")[["pct_palma_ref", "pct_no_dosel_en_palma", "ha_palma2019_sin_dosel"]]
        .mean().round(2))

# --- Vista de control ---
u = UBIC[c_ej]
t = int(np.argmax(u["valido"].mean(axis=(1, 2))))
base = rgb_escena(u, t)
capa = base.copy()
capa[u["palma_ref"]] = 0.5 * capa[u["palma_ref"]] + 0.5 * np.array([0, 1, 0])
capa[u["no_dosel"] & u["palma_ref"]] = [1, 0.55, 0]
capa[~u["valido"][t]] = [1, 1, 1]
fig, ax = plt.subplots(1, 4, figsize=(24, 6))
ax[0].imshow(base); ax[0].set_title(f"{c_ej} · {u['fechas'][t]}")
ax[1].imshow(np.isin(u["gop"], (1, 2)), cmap="Greens"); ax[1].set_title(f"Etiqueta ({u['fuente_etiqueta']})")
ax[2].imshow(prob_palma(unet, u, t), cmap="Greens", vmin=0, vmax=1); ax[2].set_title("U-Net: P(palma) en t")
ax[3].imshow(capa); ax[3].set_title("verde = palma de referencia · naranja = no-dosel · blanco = nube")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

# --- Exportación de máscaras ---
t0 = time.time()
for c, u in UBIC.items():
    escribir_tif(os.path.join(OUT_DIR, "modulo2_mascaras", c, f"{c}_mascaras_ubicacion.tif"),
                 np.stack([u["palma_ref"], u["no_dosel"], np.isin(u["gop"], (1, 2)), u["palma2019_sin_dosel"]]),
                 u, "uint8", None, ["palma_referencia", "no_dosel_persistente", "etiqueta_palma", "palma2019_sin_dosel"])
    for t, fecha in enumerate(u["fechas"]):
        escribir_tif(os.path.join(OUT_DIR, "modulo2_mascaras", c, f"{fecha}_mascaras.tif"),
                     np.stack([u["palma_t"][t], u["valido"][t], u["evaluable"][t]]),
                     u, "uint8", None, ["palma_t", "valido", "evaluable"])
print(f"Máscaras exportadas en {OUT_DIR}/modulo2_mascaras ({time.time() - t0:.0f} s)")


# MÓDULO 3 — Detección de anomalías
**Entrada:** {X, índices, máscaras}. **Salida:** anomalía(t) por detector + votos de consenso.

Todos los detectores ven lo mismo: los **14 canales normalizados** ({X, índices}) de cada píxel **evaluable**.
Los clásicos usan el vector del píxel; los profundos usan además su vecindario (parche de 9×9 = 90 m).

| Detector | Tipo | Puntaje de anomalía |
|---|---|---|
| One-Class SVM | clásico | −función de decisión (frontera RBF alrededor de lo normal) |
| Isolation Forest | clásico | cuán fácil es aislar el píxel con cortes aleatorios |
| Local Outlier Factor | clásico | densidad local relativa a sus vecinos |
| **SimCLR** | autosupervisado contrastivo | Mahalanobis al "diccionario" de lo normal en el espacio aprendido (SSD) |
| **BYOL** | autosupervisado sin negativos | igual que SimCLR |
| **Autoencoder** | autosupervisado reconstructivo (idea del paper NNDLR) | error de reconstrucción del píxel central |

**Protocolo común:**
1. Se entrenan con píxeles evaluables de las ubicaciones de **train**.
2. El umbral de cada uno es el percentil `1 − CONTAMINATION` de sus puntajes sobre píxeles de **validación**.
3. Se evalúan en **test** con anomalías sintéticas y se aplican a todas las escenas.

## 3.1 Muestreo de píxeles de entrenamiento, calibración y evaluación
Los parches se recortan en la GPU directamente de la escena (con relleno por reflexión en los bordes). Se guardan
en reflectancia: los índices se recalculan después de cada aumentación para que sigan siendo coherentes.


In [ ]:
R = PATCH // 2
_OFF = torch.arange(PATCH, device=DEVICE)
_IB = {n: k for k, n in enumerate(BANDAS)}


def bandas_gpu(u, t):
    """Bandas de la escena en GPU con relleno por reflexión de R píxeles: (10, H+2R, W+2R)."""
    x = torch.from_numpy(u["bandas"][t].astype(np.float32)).to(DEVICE)[None]
    return F.pad(x, (R, R, R, R), mode="reflect")[0]


def recortar(xpad, i, j):
    """Parches (N, 10, P, P) centrados en los píxeles (i, j) de la escena original."""
    fil = (i[:, None] + _OFF)[:, :, None]
    col = (j[:, None] + _OFF)[:, None, :]
    return xpad[:, fil, col].permute(1, 0, 2, 3)


def canales(bandas):
    """X (N, 10, P, P) -> {X, índices} normalizado (N, 14, P, P), en GPU."""
    b = lambda n: bandas[:, _IB[n]]
    nd = lambda a, c: ((a - c) / (a + c + 1e-6)).clamp(-1, 1)
    idx = torch.stack([nd(b("B08"), b("B04")), nd(b("B08"), b("B03")),
                       nd(b("B08"), b("B05")), nd(b("B08"), b("B11"))], dim=1)
    return (torch.cat([bandas, idx], dim=1) - MU_T) / SD_T


def muestrear(ids, n_total, rng):
    """(crop_id, t, i, j) de píxeles evaluables, repartidos parejo entre todas las escenas de 'ids'."""
    escenas = [(c, t) for c in ids for t in range(len(UBIC[c]["fechas"])) if UBIC[c]["evaluable"][t].any()]
    por_escena = int(np.ceil(n_total / max(1, len(escenas))))
    partes = []
    for c, t in escenas:
        ii, jj = np.nonzero(UBIC[c]["evaluable"][t])
        s = rng.choice(len(ii), min(por_escena, len(ii)), replace=False)
        partes.append(pd.DataFrame(dict(crop_id=c, t=t, i=ii[s], j=jj[s])))
    df = pd.concat(partes, ignore_index=True)
    if len(df) > n_total:
        df = df.sample(n_total, random_state=rng.randint(2 ** 31 - 1)).reset_index(drop=True)
    return df


@torch.no_grad()
def extraer_parches(df):
    salida = torch.empty((len(df), len(BANDAS), PATCH, PATCH), dtype=torch.float32)
    for (c, t), g in df.groupby(["crop_id", "t"]):
        xpad = bandas_gpu(UBIC[c], t)
        i = torch.as_tensor(g.i.values, device=DEVICE)
        j = torch.as_tensor(g.j.values, device=DEVICE)
        salida[torch.as_tensor(g.index.values)] = recortar(xpad, i, j).cpu()
    return salida


t0 = time.time()
muestras_train = muestrear(IDS["train"], N_PARCHES_ENTRENAMIENTO, rng_para("muestras_train"))
muestras_cal = muestrear(IDS["val"], N_CALIBRACION, rng_para("muestras_cal"))
muestras_eval = muestrear(IDS["test"], N_EVALUACION, rng_para("muestras_eval"))
P_TRAIN, P_CAL, P_EVAL = (extraer_parches(d) for d in (muestras_train, muestras_cal, muestras_eval))
idx_ref = rng_para("referencia").choice(len(P_TRAIN), min(N_REFERENCIA, len(P_TRAIN)), replace=False)
P_REF = P_TRAIN[torch.as_tensor(idx_ref)]
print(f"Parches: entrenamiento {tuple(P_TRAIN.shape)} | referencia {len(P_REF)} | "
      f"calibración (val) {len(P_CAL)} | evaluación (test) {len(P_EVAL)}  ({time.time() - t0:.0f} s)")


## 3.2 Detectores clásicos: One-Class SVM, Isolation Forest, LOF
Se ajustan sobre el vector de 14 canales del píxel, usando el banco de referencia (train). El One-Class SVM usa un
subconjunto de 10.000 píxeles, porque su entrenamiento crece de forma aproximadamente cuadrática.


In [ ]:
@torch.no_grad()
def rasgos_centro(P):
    """Vector de 14 canales normalizados del píxel central de cada parche."""
    out = []
    for s in range(0, len(P), LOTE_INFERENCIA):
        out.append(canales(P[s:s + LOTE_INFERENCIA].to(DEVICE))[:, :, R, R].cpu().numpy())
    return np.concatenate(out).astype(np.float64)


t0 = time.time()
X_ref = rasgos_centro(P_REF)
ocsvm = OneClassSVM(kernel="rbf", nu=CONTAMINATION, gamma="scale").fit(X_ref[:N_REFERENCIA_OCSVM])
iforest = IsolationForest(n_estimators=300, contamination="auto", random_state=SEED, n_jobs=-1).fit(X_ref)
lof = LocalOutlierFactor(n_neighbors=35, novelty=True, n_jobs=-1).fit(X_ref)
for nombre, m in [("ocsvm", ocsvm), ("isolation_forest", iforest), ("lof", lof)]:
    joblib.dump(m, os.path.join(MODELOS_DIR, f"{nombre}.joblib"))
print(f"Clásicos entrenados con {len(X_ref)} píxeles de referencia en {time.time() - t0:.0f} s "
      f"(OCSVM: {ocsvm.support_vectors_.shape[0]} vectores de soporte).")


## 3.3 SimCLR y BYOL
Idea común: dos aumentaciones aleatorias del mismo parche (dos "vistas") deben dar representaciones parecidas.

- **SimCLR** (Chen et al., 2020). El encoder `f` produce la representación `h` y la cabeza `g` (MLP de 2 capas con
  BN) produce `z`. La pérdida es **NT-Xent** con temperatura τ: cada vista debe reconocer a su pareja entre las
  `2(N−1)` restantes del batch. Está portada de `objective.py` del repo oficial; la cabeza viene de
  `model_util.py` y el learning rate usa warmup lineal + coseno como `learning_rate_schedule()`.
- **BYOL** (Grill et al., 2020). La red *online* (`f`, `g` y un predictor `q`) predice la proyección que la red
  *target* da de la otra vista. La target es un promedio exponencial (EMA) de la online, con τ que sube de 0.996
  a 1 con curva coseno. La pérdida es `2 − 2·coseno`, simetrizada. **No usa negativos.** Al final se conserva el
  encoder online.

**Aumentaciones adaptadas a multiespectral:**
- **Geometría:** recorte aleatorio con área de 25–100% y aspecto 3/4–4/3, redimensionado, volteo y rotación
  de 90°. En una imagen cenital no hay "arriba".
- **"Color":** ganancia global ±20%, ganancia por banda ±5% y ruido leve, con p = 0.8. Es **suave a
  propósito**: la forma de la firma espectral es justo la señal de estrés, y una distorsión fuerte enseñaría al
  modelo a ignorarla. Por la misma razón **no se usa escala de grises**.
- **Desenfoque:** SimCLR no lo usa (el repo lo desactiva en imágenes pequeñas). BYOL lo aplica asimétrico: p = 1.0
  en una vista y 0.1 en la otra, como en su paper.

**Diferencias de escala respecto al paper, no de concepto:**
- El encoder es una ResNet pequeña. La salida `h` concatena el promedio del parche con el **píxel central**, para
  que la representación no pierda precisión espacial.
- Se usa AdamW en vez de LARS; LARS es para batches de 4096+.
- Las dos vistas pasan juntas por la red, así que la BatchNorm ve todo el batch (la "BN global" del paper en una
  sola GPU).

**De representación a puntaje (SSD).** Sehwag et al. (ICLR 2021) proponen, para SimCLR, **k-means sobre la
representación del conjunto normal + distancia de Mahalanobis al cluster más cercano**. Es equivalente a un
*diccionario del fondo*, como el del paper NNDLR, y escala a las decenas de millones de píxeles de este dataset,
cosa que un k-NN contra un banco de 20.000 vectores no permitiría. Se usa `h`, no `z`: según el paper de SimCLR,
la capa previa a la proyección conserva más información.


In [ ]:
# ============================================================
# CONFIGURACIÓN SSL + AUMENTACIONES
# ============================================================
EPOCAS_SSL, EPOCAS_WARMUP_SSL, BATCH_SIZE_SSL = 50, 5, 512
LR_SSL, WEIGHT_DECAY_SSL = 1e-3, 1e-6        # el paper de SimCLR usa weight decay 1e-6
ANCHO_ENCODER, DIM_PROYECCION = 64, 128      # h tiene 2*ANCHO_ENCODER = 128 dimensiones
TEMPERATURA_SIMCLR = 0.5     # el paper usa 0.1 con batch 4096; el repo usa 0.5 con batch 512 (CIFAR-10)
PROB_BLUR_SIMCLR = 0.0
TAU_BASE_BYOL, DIM_OCULTA_BYOL, PROB_BLUR_BYOL = 0.996, 512, (1.0, 0.1)
AREA_MIN_RECORTE, FUERZA_JITTER_ESPECTRAL = 0.25, 0.5
N_CLUSTERS_SSD = 10
LATENTE_AE, EPOCAS_AE = 32, 40


def rotar_voltear(x):
    """Volteo horizontal (p = 0.5) + rotación k*90° por muestra."""
    n, dev = x.shape[0], x.device
    x = torch.where((torch.rand(n, device=dev) < 0.5)[:, None, None, None], x.flip(-1), x)
    k = torch.randint(0, 4, (n,), device=dev)
    return torch.stack([torch.rot90(x, r, dims=(2, 3)) for r in range(4)])[k, torch.arange(n, device=dev)]


def _recorte_geometrico(x):
    n, dev = x.shape[0], x.device
    area = torch.empty(n, device=dev).uniform_(AREA_MIN_RECORTE, 1.0)
    aspecto = torch.exp(torch.empty(n, device=dev).uniform_(math.log(3 / 4), math.log(4 / 3)))
    sx, sy = torch.sqrt(area * aspecto).clamp(max=1.0), torch.sqrt(area / aspecto).clamp(max=1.0)
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 1, 1] = sx, sy
    theta[:, 0, 2] = (torch.rand(n, device=dev) * 2 - 1) * (1 - sx)
    theta[:, 1, 2] = (torch.rand(n, device=dev) * 2 - 1) * (1 - sy)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    return rotar_voltear(x)


def _jitter_espectral(x, prob=0.8):
    n, b, dev = x.shape[0], x.shape[1], x.device
    f = FUERZA_JITTER_ESPECTRAL
    g_global = 1 + (torch.rand(n, 1, 1, 1, device=dev) * 2 - 1) * 0.4 * f      # ±20% con f = 0.5
    g_banda = 1 + (torch.rand(n, b, 1, 1, device=dev) * 2 - 1) * 0.1 * f       # ±5%
    ruido = torch.randn_like(x) * SD_T[:, :b] * 0.05 * f
    return torch.where(torch.rand(n, 1, 1, 1, device=dev) < prob, x * g_global * g_banda + ruido, x)


def _desenfoque(x, prob):
    if prob <= 0:
        return x
    n, c, h, w = x.shape
    dev = x.device
    sigma = torch.empty(n, device=dev).uniform_(0.1, 1.0)
    pos = torch.tensor([-1.0, 0.0, 1.0], device=dev)
    k1 = torch.exp(-(pos[None, :] ** 2) / (2 * sigma[:, None] ** 2))
    k1 = k1 / k1.sum(dim=1, keepdim=True)
    pesos = (k1[:, :, None] * k1[:, None, :])[:, None].repeat_interleave(c, dim=0)   # (N*C, 1, 3, 3)
    xp = F.pad(x.reshape(1, n * c, h, w), (1, 1, 1, 1), mode="reflect")
    x_b = F.conv2d(xp, pesos, groups=n * c).reshape(n, c, h, w)
    return torch.where((torch.rand(n, device=dev) < prob)[:, None, None, None], x_b, x)


def vista(x, prob_blur):
    """Una vista t ~ T: aumenta la reflectancia y DESPUÉS calcula índices y normaliza."""
    return canales(_desenfoque(_jitter_espectral(_recorte_geometrico(x)), prob_blur))


In [ ]:
# ============================================================
# ENCODER, CABEZAS, PÉRDIDAS Y ENTRENAMIENTO (SimCLR / BYOL)
# ============================================================
class BloqueResidual(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.atajo = nn.Identity() if c_in == c_out else nn.Sequential(
            nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(y)) + self.atajo(x))


class EncoderEspectroEspacial(nn.Module):
    """f(.): conv 1x1 (mezcla espectral por píxel) + 2 bloques residuales 3x3 (contexto espacial).
    h = [promedio del parche ; vector del píxel central] -> 2*ancho dimensiones."""
    def __init__(self, c_in=len(CANALES), ancho=ANCHO_ENCODER):
        super().__init__()
        self.espectral = nn.Sequential(nn.Conv2d(c_in, ancho, 1, bias=False), nn.BatchNorm2d(ancho),
                                       nn.ReLU(inplace=True))
        self.bloque1, self.bloque2 = BloqueResidual(ancho, ancho), BloqueResidual(ancho, ancho)
        self.dim_salida = 2 * ancho

    def forward(self, x):
        y = self.bloque2(self.bloque1(self.espectral(x)))
        c = y.shape[-1] // 2
        return torch.cat([y.mean(dim=(2, 3)), y[:, :, c, c]], dim=1)


def cabeza_simclr(d_in, d_out=DIM_PROYECCION):
    """g(.): MLP no lineal de 2 capas con BN; capa intermedia del tamaño de h (projection_head() del repo)."""
    return nn.Sequential(nn.Linear(d_in, d_in, bias=False), nn.BatchNorm1d(d_in), nn.ReLU(inplace=True),
                         nn.Linear(d_in, d_out, bias=False), nn.BatchNorm1d(d_out))


def mlp_byol(d_in, d_oculta=DIM_OCULTA_BYOL, d_out=DIM_PROYECCION):
    """Proyector y predictor de BYOL: Linear -> BN -> ReLU -> Linear."""
    return nn.Sequential(nn.Linear(d_in, d_oculta), nn.BatchNorm1d(d_oculta), nn.ReLU(inplace=True),
                         nn.Linear(d_oculta, d_out))


LARGE_NUM = 1e9


def perdida_nt_xent(z1, z2, temperatura):
    """NT-Xent: port de add_contrastive_loss() (objective.py, google-research/simclr)."""
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    n = z1.shape[0]
    etiquetas = torch.arange(n, device=z1.device)
    mascara = torch.eye(n, device=z1.device) * LARGE_NUM
    logits_aa = z1 @ z1.T / temperatura - mascara
    logits_bb = z2 @ z2.T / temperatura - mascara
    logits_ab = z1 @ z2.T / temperatura
    logits_ba = z2 @ z1.T / temperatura
    perdida = (F.cross_entropy(torch.cat([logits_ab, logits_aa], 1), etiquetas)
               + F.cross_entropy(torch.cat([logits_ba, logits_bb], 1), etiquetas))
    return perdida, (logits_ab.argmax(1) == etiquetas).float().mean()


def perdida_byol(p, z):
    """2 - 2*coseno(p, z); z (red target) sin gradiente."""
    return (2 - 2 * F.cosine_similarity(p, z.detach(), dim=-1)).mean()


@torch.no_grad()
def actualizar_ema(online, target, tau):
    for p_o, p_t in zip(online.parameters(), target.parameters()):
        p_t.mul_(tau).add_(p_o.detach(), alpha=1 - tau)
    for b_o, b_t in zip(online.buffers(), target.buffers()):
        b_t.copy_(b_o)


def _factor_lr(paso, total, warmup):
    if paso < warmup:
        return (paso + 1) / warmup
    return 0.5 * (1 + math.cos(math.pi * (paso - warmup) / max(1, total - warmup)))


def preparar_optimizador(parametros, n_datos, epocas, epocas_warmup):
    bs = min(BATCH_SIZE_SSL, n_datos)
    pasos = max(1, n_datos // bs)
    total, warmup = pasos * epocas, pasos * epocas_warmup
    opt = torch.optim.AdamW(parametros, lr=LR_SSL, weight_decay=WEIGHT_DECAY_SSL)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda p: _factor_lr(p, total, warmup))
    return bs, pasos, total, opt, sched


def lotes(n, bs, gen):
    """Índices de cada batch de una época (se descarta el último incompleto: N fijo para BN y NT-Xent)."""
    perm = torch.randperm(n, generator=gen)
    for i in range(0, n - bs + 1, bs):
        yield perm[i:i + bs]


def entrenar_simclr(datos):
    torch.manual_seed(SEED)
    encoder = EncoderEspectroEspacial().to(DEVICE)
    cabeza = cabeza_simclr(encoder.dim_salida).to(DEVICE)
    bs, pasos, total, opt, sched = preparar_optimizador(
        list(encoder.parameters()) + list(cabeza.parameters()), len(datos), EPOCAS_SSL, EPOCAS_WARMUP_SSL)
    gen, hist, t0 = torch.Generator().manual_seed(SEED), {"pérdida NT-Xent": [], "acc. contrastiva": []}, time.time()
    for ep in range(EPOCAS_SSL):
        encoder.train(); cabeza.train()
        suma_l = suma_a = 0.0
        for idx in lotes(len(datos), bs, gen):
            x = datos[idx.to(datos.device)]
            v = torch.cat([vista(x, PROB_BLUR_SIMCLR), vista(x, PROB_BLUR_SIMCLR)])   # 2N vistas juntas
            z1, z2 = cabeza(encoder(v)).chunk(2)
            perdida, acc = perdida_nt_xent(z1, z2, TEMPERATURA_SIMCLR)
            opt.zero_grad(set_to_none=True); perdida.backward(); opt.step(); sched.step()
            suma_l += perdida.item(); suma_a += acc.item()
        hist["pérdida NT-Xent"].append(suma_l / pasos); hist["acc. contrastiva"].append(suma_a / pasos)
        if ep == 0 or (ep + 1) % 10 == 0:
            print(f"  SimCLR época {ep + 1:3d}/{EPOCAS_SSL} | pérdida {hist['pérdida NT-Xent'][-1]:.4f} | "
                  f"acc. contrastiva {hist['acc. contrastiva'][-1]:.3f} | {time.time() - t0:.0f} s")
    return encoder.eval(), hist


def entrenar_byol(datos):
    torch.manual_seed(SEED)
    enc_o = EncoderEspectroEspacial().to(DEVICE)
    proy_o, pred = mlp_byol(enc_o.dim_salida).to(DEVICE), mlp_byol(DIM_PROYECCION).to(DEVICE)
    enc_t, proy_t = copy.deepcopy(enc_o), copy.deepcopy(proy_o)
    for p in list(enc_t.parameters()) + list(proy_t.parameters()):
        p.requires_grad_(False)
    bs, pasos, total, opt, sched = preparar_optimizador(
        list(enc_o.parameters()) + list(proy_o.parameters()) + list(pred.parameters()),
        len(datos), EPOCAS_SSL, EPOCAS_WARMUP_SSL)
    gen, hist = torch.Generator().manual_seed(SEED), {"pérdida BYOL": [], "tau (EMA)": []}
    paso, tau, t0 = 0, TAU_BASE_BYOL, time.time()
    for ep in range(EPOCAS_SSL):
        for m in (enc_o, proy_o, pred, enc_t, proy_t):
            m.train()
        suma = 0.0
        for idx in lotes(len(datos), bs, gen):
            x = datos[idx.to(datos.device)]
            v = torch.cat([vista(x, PROB_BLUR_BYOL[0]), vista(x, PROB_BLUR_BYOL[1])])
            p1, p2 = pred(proy_o(enc_o(v))).chunk(2)
            with torch.no_grad():
                t1, t2 = proy_t(enc_t(v)).chunk(2)
            perdida = perdida_byol(p1, t2) + perdida_byol(p2, t1)   # cada vista predice a la otra
            opt.zero_grad(set_to_none=True); perdida.backward(); opt.step(); sched.step()
            tau = 1 - (1 - TAU_BASE_BYOL) * (math.cos(math.pi * paso / total) + 1) / 2
            actualizar_ema(enc_o, enc_t, tau); actualizar_ema(proy_o, proy_t, tau)
            paso += 1
            suma += perdida.item()
        hist["pérdida BYOL"].append(suma / pasos); hist["tau (EMA)"].append(tau)
        if ep == 0 or (ep + 1) % 10 == 0:
            print(f"  BYOL   época {ep + 1:3d}/{EPOCAS_SSL} | pérdida {hist['pérdida BYOL'][-1]:.4f} | "
                  f"tau {tau:.4f} | {time.time() - t0:.0f} s")
    return enc_o.eval(), hist


def graficar_historial(hist, titulo, ruta_png=None):
    fig, axs = plt.subplots(1, len(hist), figsize=(5 * len(hist), 3.3))
    for ax, (nombre, valores) in zip(np.atleast_1d(axs), hist.items()):
        ax.plot(range(1, len(valores) + 1), valores); ax.set_title(nombre); ax.set_xlabel("época"); ax.grid(alpha=0.3)
    fig.suptitle(titulo); fig.tight_layout()
    if ruta_png:
        fig.savefig(ruta_png, dpi=110, bbox_inches="tight")
    plt.show()


def cargar_o_entrenar(nombre_archivo, constructor, entrenar, titulo):
    ruta = ruta_existente("modelos", nombre_archivo)
    if CARGAR_SI_EXISTE and ruta:
        modelo = constructor().to(DEVICE)
        modelo.load_state_dict(torch.load(ruta, map_location=DEVICE))
        print(f"{titulo}: cargado de {ruta}")
        return modelo.eval()
    print(f"Entrenando {titulo}...")
    modelo, hist = entrenar()
    torch.save(modelo.state_dict(), os.path.join(MODELOS_DIR, nombre_archivo))
    graficar_historial(hist, titulo, os.path.join(OUT_DIR, f"curva_{nombre_archivo.split('.')[0]}.png"))
    return modelo


DATOS_GPU = P_TRAIN.to(DEVICE)
print(f"Datos de entrenamiento en {DEVICE}: {tuple(DATOS_GPU.shape)}")


### Entrenamiento de SimCLR
La *acc. contrastiva* es el % de veces que una vista encuentra a su pareja entre todas las del batch. Debe subir,
pero el paper advierte que por sí sola no mide la calidad de la representación.


In [ ]:
encoder_simclr = cargar_o_entrenar("encoder_simclr.pt", EncoderEspectroEspacial,
                                   lambda: entrenar_simclr(DATOS_GPU), "SimCLR")


### Entrenamiento de BYOL
La pérdida de BYOL no tiene un valor "bueno" absoluto. Lo que hay que vigilar es el **colapso** (todas las
representaciones iguales); se revisa en la sección 3.5.


In [ ]:
encoder_byol = cargar_o_entrenar("encoder_byol.pt", EncoderEspectroEspacial,
                                 lambda: entrenar_byol(DATOS_GPU), "BYOL")


## 3.4 Autoencoder convolucional (idea del paper NNDLR / notebook anterior)
Reconstruye el parche de 14 canales pasando por un cuello de botella de 32 dimensiones. Como solo vio palma
normal, reconstruye mal lo que no se le parece. El puntaje es el **error de reconstrucción del píxel central**:
así el mapa queda a resolución de píxel, en vez de promediar parches de 32×32 como en la versión anterior. Se
entrena con rotaciones y volteos, y guarda el modelo con menor error en validación.


In [ ]:
class AutoencoderParches(nn.Module):
    def __init__(self, c_in=len(CANALES), ancho=64, latente=LATENTE_AE, p=PATCH):
        super().__init__()
        self.ancho, self.p = ancho, p
        bloque = lambda i, o, k: [nn.Conv2d(i, o, k, padding=k // 2, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]
        self.enc = nn.Sequential(*bloque(c_in, ancho, 1), *bloque(ancho, ancho, 3), *bloque(ancho, ancho, 3))
        self.a_latente = nn.Linear(ancho * p * p, latente)
        self.desde_latente = nn.Sequential(nn.Linear(latente, ancho * p * p), nn.ReLU(inplace=True))
        self.dec = nn.Sequential(*bloque(ancho, ancho, 3), *bloque(ancho, ancho, 3), nn.Conv2d(ancho, c_in, 1))

    def forward(self, x):
        z = self.a_latente(self.enc(x).flatten(1))
        return self.dec(self.desde_latente(z).view(-1, self.ancho, self.p, self.p)), z


def entrenar_autoencoder(datos, datos_val):
    torch.manual_seed(SEED)
    ae = AutoencoderParches().to(DEVICE)
    bs, pasos, total, opt, sched = preparar_optimizador(ae.parameters(), len(datos), EPOCAS_AE, 2)
    gen, t0 = torch.Generator().manual_seed(SEED), time.time()
    hist, mejor, estado = {"MSE entrenamiento": [], "MSE validación": []}, float("inf"), None
    for ep in range(EPOCAS_AE):
        ae.train()
        suma = 0.0
        for idx in lotes(len(datos), bs, gen):
            x = canales(rotar_voltear(datos[idx.to(datos.device)]))
            perdida = F.mse_loss(ae(x)[0], x)
            opt.zero_grad(set_to_none=True); perdida.backward(); opt.step(); sched.step()
            suma += perdida.item()
        ae.eval()
        with torch.no_grad():
            val = np.mean([F.mse_loss(ae(x)[0], x).item()
                           for x in (canales(datos_val[s:s + LOTE_INFERENCIA])
                                     for s in range(0, len(datos_val), LOTE_INFERENCIA))])
        hist["MSE entrenamiento"].append(suma / pasos); hist["MSE validación"].append(val)
        if val < mejor:
            mejor, estado = val, copy.deepcopy(ae.state_dict())
        if ep == 0 or (ep + 1) % 10 == 0:
            print(f"  AE época {ep + 1:3d}/{EPOCAS_AE} | MSE train {suma / pasos:.4f} | MSE val {val:.4f} | "
                  f"{time.time() - t0:.0f} s")
    ae.load_state_dict(estado)
    return ae.eval(), hist


autoencoder = cargar_o_entrenar("autoencoder.pt", AutoencoderParches,
                                lambda: entrenar_autoencoder(DATOS_GPU, P_CAL[:5000].to(DEVICE)), "Autoencoder")


## 3.5 Puntajes, calibración de umbrales y diagnóstico de colapso
- **SimCLR / BYOL:** representación `h` normalizada → k-means (10 clusters) sobre el banco de referencia →
  covarianza de cada cluster con encogimiento de Ledoit-Wolf → puntaje = Mahalanobis² al cluster más cercano (SSD).
- **Autoencoder:** error cuadrático medio del píxel central (14 canales).
- **Clásicos:** `−score_samples` de scikit-learn (más alto = más anómalo).

Todos los umbrales salen del **mismo** criterio: el percentil `1 − CONTAMINATION` de los puntajes en píxeles de
**validación**.


In [ ]:
class PuntuadorSSD:
    """SSD (Sehwag et al., 2021): k-means en el espacio de representación + Mahalanobis al cluster más cercano."""
    def __init__(self, n_clusters=N_CLUSTERS_SSD):
        self.n_clusters = n_clusters

    def ajustar(self, H):
        etiquetas = KMeans(n_clusters=self.n_clusters, n_init=5, random_state=SEED).fit_predict(H)
        medias, precisiones = [], []
        for k in range(self.n_clusters):
            Hk = H[etiquetas == k]
            if len(Hk) < 50:
                continue
            lw = LedoitWolf().fit(Hk)
            medias.append(lw.location_); precisiones.append(lw.precision_)
        self.medias = torch.tensor(np.stack(medias), dtype=torch.float32, device=DEVICE)          # (K, d)
        self.precisiones = torch.tensor(np.stack(precisiones), dtype=torch.float32, device=DEVICE)  # (K, d, d)
        self.tamanos = np.bincount(etiquetas, minlength=self.n_clusters)
        return self

    @torch.no_grad()
    def puntuar(self, h):
        dif = h.float()[:, None, :] - self.medias[None]                                  # (B, K, d)
        return torch.einsum("bkd,kde,bke->bk", dif, self.precisiones, dif).min(dim=1).values


@torch.no_grad()
def embeber(encoder, P):
    encoder.eval()
    salida = []
    for s in range(0, len(P), LOTE_INFERENCIA):
        with autocast():
            h = encoder(canales(P[s:s + LOTE_INFERENCIA].to(DEVICE)))
        salida.append(F.normalize(h.float(), dim=1).cpu())
    return torch.cat(salida).numpy()


t0 = time.time()
ssd = {}
for nombre, enc in (("SimCLR", encoder_simclr), ("BYOL", encoder_byol)):
    H_ref = embeber(enc, P_REF)
    ssd[nombre] = PuntuadorSSD().ajustar(H_ref)
    diversidad, referencia = H_ref.std(0).mean(), 1 / math.sqrt(H_ref.shape[1])
    print(f"{nombre}: diversidad de h = {diversidad:.4f} (≈{referencia:.4f} = bien repartida; ~0 = colapso) | "
          f"tamaños de cluster: {ssd[nombre].tamanos.tolist()}")
    if diversidad < 0.1 * referencia:
        print(f"  ⚠️  {nombre}: la representación parece colapsada; prueba otro LR_SSL o más épocas.")

DETECTORES = {
    "One-Class SVM": ("clasico", ocsvm),
    "Isolation Forest": ("clasico", iforest),
    "LOF": ("clasico", lof),
    "SimCLR": ("ssl", (encoder_simclr, ssd["SimCLR"])),
    "BYOL": ("ssl", (encoder_byol, ssd["BYOL"])),
    "Autoencoder": ("ae", autoencoder),
}
NOMBRES_DET = list(DETECTORES)


@torch.no_grad()
def puntuar_lote(x):
    """x: {X, índices} normalizado (B, 14, P, P) en GPU -> {detector: puntaje (B,)}; más alto = más anómalo."""
    salida = {}
    centro = x[:, :, R, R].float().cpu().numpy().astype(np.float64)
    for nombre, (tipo, m) in DETECTORES.items():
        if tipo == "clasico":
            salida[nombre] = -m.score_samples(centro)
        elif tipo == "ssl":
            enc, puntuador = m
            with autocast():
                h = enc(x)
            salida[nombre] = puntuador.puntuar(F.normalize(h.float(), dim=1)).cpu().numpy()
        else:
            with autocast():
                rec = m(x)[0]
            salida[nombre] = ((rec.float() - x)[:, :, R, R] ** 2).mean(1).cpu().numpy()
    return salida


def puntuar_parches(P):
    acum = {m: [] for m in NOMBRES_DET}
    for s in range(0, len(P), LOTE_INFERENCIA):
        for m, v in puntuar_lote(canales(P[s:s + LOTE_INFERENCIA].to(DEVICE))).items():
            acum[m].append(v)
    return {m: np.concatenate(v) for m, v in acum.items()}


S_CAL = puntuar_parches(P_CAL)
UMBRALES = {m: float(np.quantile(s, 1 - CONTAMINATION)) for m, s in S_CAL.items()}
S_EVAL = puntuar_parches(P_EVAL)
tabla_umbral = pd.DataFrame({
    "umbral": UMBRALES,
    "% marcado en validación": {m: 100 * (S_CAL[m] > UMBRALES[m]).mean() for m in NOMBRES_DET},
    "% marcado en test": {m: 100 * (S_EVAL[m] > UMBRALES[m]).mean() for m in NOMBRES_DET},
})
print(f"Calibración lista en {time.time() - t0:.0f} s. En validación cada detector marca ~{100 * CONTAMINATION:.0f}% "
      f"por construcción; en test debería ser parecido si los lotes nuevos son similares:")
display(tabla_umbral.round(4))


## 3.6 Evaluación con anomalías sintéticas de estrés (ubicaciones de test)
Sin etiquetas de campo no se puede medir la precisión real. Como *proxy*, a píxeles normales de ubicaciones que
**ningún modelo vio** se les inyecta en el centro del parche (5×5 px = 50 m) una firma de estrés vegetal
genérica, con severidad `s`:
- **Menos NIR:** B07, B08 y B8A × (1 − s). Hay menos estructura foliar.
- **Más rojo:** B04 × (1 + s) y B05 × (1 + s/2). Hay menos clorofila.
- **Más SWIR:** B11 y B12 × (1 + s). Hay menos agua en el dosel.
- **Algo más de visible:** B02 y B03 × (1 + s/2). B06 baja × (1 − s/2).

Se reportan tres cosas:
- **AUROC:** capacidad de separar sano de estresado, sin depender del umbral.
- **TPR:** % de píxeles estresados que se detectan con el umbral calibrado.
- **FPR:** % de píxeles sanos marcados.

Esto mide la **sensibilidad a un cambio espectral plausible**; no reemplaza la validación en campo.


In [ ]:
FIRMA_ESTRES = {"B07": -1.0, "B08": -1.0, "B8A": -1.0, "B04": +1.0, "B05": +0.5,
                "B11": +1.0, "B12": +1.0, "B02": +0.5, "B03": +0.5, "B06": -0.5}


def inyectar_estres(P, severidad, lado=5):
    Q = P.clone()
    a = R - lado // 2
    for banda, signo in FIRMA_ESTRES.items():
        Q[:, _IB[banda], a:a + lado, a:a + lado] *= (1 + signo * severidad)
    return Q


def ndvi_centro(P):
    b8, b4 = P[:, _IB["B08"], R, R], P[:, _IB["B04"], R, R]
    return ((b8 - b4) / (b8 + b4 + 1e-6)).numpy()


t0 = time.time()
filas = []
votos_normal = sum((S_EVAL[m] > UMBRALES[m]).astype(int) for m in NOMBRES_DET)
for sev in (0.10, 0.20, 0.30):
    P_AN = inyectar_estres(P_EVAL, sev)
    S_AN = puntuar_parches(P_AN)
    delta_ndvi = float(np.mean(ndvi_centro(P_AN) - ndvi_centro(P_EVAL)))
    y = np.r_[np.zeros(len(P_EVAL)), np.ones(len(P_AN))]
    for m in NOMBRES_DET:
        filas.append(dict(detector=m, severidad=sev, dNDVI=delta_ndvi,
                          AUROC=roc_auc_score(y, np.r_[S_EVAL[m], S_AN[m]]),
                          TPR=100 * (S_AN[m] > UMBRALES[m]).mean(), FPR=100 * (S_EVAL[m] > UMBRALES[m]).mean()))
    votos_an = sum((S_AN[m] > UMBRALES[m]).astype(int) for m in NOMBRES_DET)
    filas.append(dict(detector=f"Consenso (>= {CONSENSO_MIN})", severidad=sev, dNDVI=delta_ndvi,
                      AUROC=roc_auc_score(y, np.r_[votos_normal, votos_an]),
                      TPR=100 * (votos_an >= CONSENSO_MIN).mean(), FPR=100 * (votos_normal >= CONSENSO_MIN).mean()))
eval_sintetica = pd.DataFrame(filas)
eval_sintetica.to_csv(os.path.join(OUT_DIR, "evaluacion_sintetica.csv"), index=False)
print(f"Evaluación sintética en {time.time() - t0:.0f} s (ΔNDVI medio del píxel central por severidad: "
      f"{eval_sintetica.groupby('severidad').dNDVI.first().round(3).to_dict()})")
display(eval_sintetica.pivot(index="detector", columns="severidad", values=["AUROC", "TPR"]).round(3)
        .loc[NOMBRES_DET + [f"Consenso (>= {CONSENSO_MIN})"]])

fig, ax = plt.subplots(figsize=(10, 4))
piv = eval_sintetica.pivot(index="detector", columns="severidad", values="AUROC").loc[
    NOMBRES_DET + [f"Consenso (>= {CONSENSO_MIN})"]]
piv.plot.bar(ax=ax, rot=20)
ax.set_ylabel("AUROC"); ax.set_ylim(0.4, 1.0); ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_title("Detección de estrés sintético en ubicaciones de test"); ax.legend(title="severidad")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "evaluacion_sintetica.png"), dpi=110); plt.show()


## 3.7 Inferencia en todas las escenas (salida del Módulo 3)
Para cada ubicación y fecha se evalúan los 6 detectores en todos los píxeles `evaluable(t)`. Se guarda un arreglo
de 7 capas:
- **Capas 1 a 6:** la máscara de cada detector (1 = anómalo, 0 = normal, 255 = no evaluado).
- **Capa 7:** los **votos**, es decir, cuántos detectores marcaron el píxel.

Un píxel es **anomalía de consenso** si tiene `votos ≥ CONSENSO_MIN`.

Además, por ubicación se calcula la **persistencia**: la fracción de sus fechas evaluadas en que el píxel fue
anomalía de consenso. Una anomalía que aparece y se repite en fechas sucesivas es mucho más creíble, y más
interesante para la detección temprana, que una aislada.


In [ ]:
@torch.no_grad()
def detectar_escena(u, t):
    capas = np.full((len(NOMBRES_DET) + 1, u["H"], u["W"]), 255, np.uint8)
    ii, jj = np.nonzero(u["evaluable"][t])
    if len(ii) == 0:
        return capas
    xpad = bandas_gpu(u, t)
    acum = {m: [] for m in NOMBRES_DET}
    for s in range(0, len(ii), LOTE_INFERENCIA):
        i = torch.as_tensor(ii[s:s + LOTE_INFERENCIA], device=DEVICE)
        j = torch.as_tensor(jj[s:s + LOTE_INFERENCIA], device=DEVICE)
        for m, v in puntuar_lote(canales(recortar(xpad, i, j))).items():
            acum[m].append(v)
    votos = np.zeros(len(ii), np.uint8)
    for k, m in enumerate(NOMBRES_DET):
        anomalo = np.concatenate(acum[m]) > UMBRALES[m]
        capas[k][ii, jj] = anomalo
        votos += anomalo
    capas[-1][ii, jj] = votos
    return capas


ids_inferencia = IDS["test"] if INFERENCIA_SOLO_TEST else sorted(UBIC)
t0, filas = time.time(), []
for n_hechas, c in enumerate(ids_inferencia, start=1):
    u = UBIC[c]
    u["anomalias"] = np.stack([detectar_escena(u, t) for t in range(len(u["fechas"]))])   # (T, 7, H, W)
    votos = u["anomalias"][:, -1]
    evaluado = votos != 255
    consenso = evaluado & (votos >= CONSENSO_MIN)
    u["persistencia"] = np.where(evaluado.sum(0) > 0, consenso.sum(0) / np.maximum(evaluado.sum(0), 1), np.nan)
    for t, fecha in enumerate(u["fechas"]):
        n_ev = int(evaluado[t].sum())
        fila = dict(crop_id=c, fecha=fecha, clase=u["clase"], split=u["split"], nube_local=u["nube"][t],
                    px_evaluables=n_ev, ha_evaluables=round(n_ev * u["px_ha"], 2))
        for k, m in enumerate(NOMBRES_DET):
            fila[f"% {m}"] = 100 * int((u["anomalias"][t, k] == 1).sum()) / max(1, n_ev)
        fila["% consenso"] = 100 * int(consenso[t].sum()) / max(1, n_ev)
        fila["ha consenso"] = round(int(consenso[t].sum()) * u["px_ha"], 3)
        filas.append(fila)
    if n_hechas % 20 == 0 or n_hechas == len(ids_inferencia):
        print(f"  {n_hechas}/{len(ids_inferencia)} ubicaciones ({time.time() - t0:.0f} s)")
RESUMEN = pd.DataFrame(filas)
RESUMEN.to_csv(os.path.join(OUT_DIR, "resumen_anomalias_por_escena.csv"), index=False)
print(f"Módulo 3 listo: {len(RESUMEN)} escenas analizadas en {(time.time() - t0) / 60:.1f} min.")


## 3.8 Resultados
Tablas por clase y split, acuerdo entre detectores (índice de Jaccard), regiones anómalas con su patrón espectral,
y un visor por escena.

En la tabla de regiones, la columna `caida_principal` indica **qué índice cayó más** respecto a la palma normal
de esa misma escena, medido en desviaciones estándar (z). Ese índice sugiere hacia dónde mirar:
- **NDMI:** agua del dosel.
- **NDRE:** estrés temprano.
- **GNDVI:** clorofila.
- **NDVI:** vigor.

**No es un diagnóstico.**


In [ ]:
cols_pct = [f"% {m}" for m in NOMBRES_DET] + ["% consenso"]
print("Promedio de % de píxeles evaluables marcados, por clase y split:")
display(RESUMEN.groupby(["clase", "split"])[cols_pct].mean().round(2))

# --- Acuerdo entre detectores (Jaccard sobre todas las escenas procesadas) ---
inter = np.zeros((len(NOMBRES_DET),) * 2)
union = np.zeros_like(inter)
for c in ids_inferencia:
    A = UBIC[c]["anomalias"][:, :len(NOMBRES_DET)] == 1          # (T, 6, H, W)
    A = A.transpose(1, 0, 2, 3).reshape(len(NOMBRES_DET), -1)
    for a in range(len(NOMBRES_DET)):
        for b in range(a, len(NOMBRES_DET)):
            inter[a, b] += np.count_nonzero(A[a] & A[b])
            union[a, b] += np.count_nonzero(A[a] | A[b])
jaccard = np.triu(inter / np.maximum(union, 1)); jaccard = jaccard + np.triu(jaccard, 1).T
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(jaccard, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(NOMBRES_DET)), NOMBRES_DET, rotation=35, ha="right"); ax.set_yticks(range(len(NOMBRES_DET)), NOMBRES_DET)
for a in range(len(NOMBRES_DET)):
    for b in range(len(NOMBRES_DET)):
        ax.text(b, a, f"{jaccard[a, b]:.2f}", ha="center", va="center", color="w" if jaccard[a, b] < 0.6 else "k")
ax.set_title("Acuerdo entre detectores (Jaccard)"); plt.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "acuerdo_detectores.png"), dpi=110); plt.show()

# --- Regiones anómalas de consenso, con patrón espectral descriptivo ---
DESCRIPCION_INDICE = {"NDVI": "vigor", "GNDVI": "clorofila", "NDRE": "estrés temprano (red-edge)",
                      "NDMI": "agua del dosel"}


def regiones_escena(u, t):
    votos = u["anomalias"][t, -1]
    consenso = (votos != 255) & (votos >= CONSENSO_MIN)
    normal = (votos != 255) & ~consenso
    if not consenso.any() or normal.sum() < 30:
        return []
    ref = u["indices"][t][:, normal].astype(np.float32)
    med, desv = np.median(ref, axis=1), ref.std(axis=1) + 1e-6
    lab, _ = ndimage.label(consenso, structure=np.ones((3, 3), int))
    filas = []
    for k, sl in enumerate(ndimage.find_objects(lab), start=1):
        if sl is None:
            continue
        sel = lab[sl] == k
        n = int(sel.sum())
        if n < AREA_MIN_REGION_PX:
            continue
        z = (u["indices"][t][:, sl[0], sl[1]][:, sel].astype(np.float32).mean(axis=1) - med) / desv
        ys, xs = np.nonzero(sel)
        lon, lat = u["transform"] * (xs.mean() + sl[1].start + 0.5, ys.mean() + sl[0].start + 0.5)
        q = int(np.argmin(z))
        fila = dict(crop_id=u["crop_id"], fecha=u["fechas"][t], clase=u["clase"], split=u["split"], region=k,
                    px=n, ha=round(n * u["px_ha"], 3), lon=round(lon, 6), lat=round(lat, 6),
                    votos_medios=round(float(votos[sl][sel].mean()), 2),
                    persistencia_media=round(float(np.nanmean(u["persistencia"][sl][sel])), 2))
        fila.update({f"z_{nombre}": round(float(z[p]), 2) for p, nombre in enumerate(INDICES)})
        fila["caida_principal"] = (f"{INDICES[q]} ({DESCRIPCION_INDICE[INDICES[q]]})" if z[q] < -1
                                   else "sin caída marcada de índices (anomalía en bandas)")
        filas.append(fila)
    return filas


REGIONES = pd.DataFrame([f for c in ids_inferencia for t in range(len(UBIC[c]["fechas"]))
                         for f in regiones_escena(UBIC[c], t)])
REGIONES.to_csv(os.path.join(OUT_DIR, "regiones_anomalas.csv"), index=False)
print(f"{len(REGIONES)} regiones anómalas de consenso (>= {AREA_MIN_REGION_PX} px).")
if len(REGIONES):
    display(REGIONES.sort_values(["persistencia_media", "ha"], ascending=False).head(15))
    display(REGIONES.groupby("clase").caida_principal.value_counts().unstack(fill_value=0))


# --- Visor por escena ---
def mostrar_escena(crop_id, t=None, guardar=False):
    u = UBIC[crop_id]
    if t is None:   # por defecto, la fecha con más anomalías de consenso
        votos = u["anomalias"][:, -1]
        t = int(np.argmax(((votos != 255) & (votos >= CONSENSO_MIN)).sum(axis=(1, 2))))
    votos = u["anomalias"][t, -1]
    base = rgb_escena(u, t)
    fig, axs = plt.subplots(3, 4, figsize=(22, 16))
    axs = axs.ravel()
    axs[0].imshow(base); axs[0].set_title(f"{crop_id} ({u['clase']}, {u['split']}) · {u['fechas'][t]}")
    for k, nombre in enumerate(("NDVI", "NDRE", "NDMI"), start=1):
        axs[k].imshow(np.where(u["valido"][t], u["indices"][t][INDICES.index(nombre)].astype(np.float32), np.nan),
                      cmap="RdYlGn", vmin=-0.2, vmax=1)
        axs[k].set_title(nombre)
    ev = base.copy()
    ev[u["evaluable"][t]] = 0.5 * ev[u["evaluable"][t]] + 0.5 * np.array([0, 1, 0])
    ev[~u["valido"][t]] = [1, 1, 1]
    axs[4].imshow(ev); axs[4].set_title("Módulo 2: evaluable (verde) · nube (blanco)")
    mapa_votos = np.where(votos == 255, np.nan, votos.astype(float))
    im = axs[5].imshow(mapa_votos, cmap="magma", vmin=0, vmax=len(NOMBRES_DET))
    axs[5].set_title("Votos (nº de detectores)"); plt.colorbar(im, ax=axs[5], fraction=0.046)
    for k, nombre in enumerate(NOMBRES_DET):
        capa = base.copy()
        sel = u["anomalias"][t, k] == 1
        capa[sel] = [1, 0, 0]
        pct = 100 * sel.sum() / max(1, (votos != 255).sum())
        axs[6 + k].imshow(capa); axs[6 + k].set_title(f"{nombre}: {pct:.2f}%")
    for a in axs:
        a.axis("off")
    plt.tight_layout()
    if guardar:
        plt.savefig(os.path.join(OUT_DIR, f"vista_{crop_id}_{u['fechas'][t]}.png"), dpi=100)
    plt.show()
    cons = (votos != 255) & (votos >= CONSENSO_MIN)
    capa = base.copy(); capa[cons] = [1, 0, 0]
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(capa); ax[0].set_title(f"Anomalías de CONSENSO (>= {CONSENSO_MIN} votos)")
    im = ax[1].imshow(u["persistencia"], cmap="inferno", vmin=0, vmax=1)
    ax[1].set_title("Persistencia en el año (fracción de fechas anómalas)"); plt.colorbar(im, ax=ax[1], fraction=0.046)
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()


# ejemplo: la ubicación de test con más hectáreas de consenso
fila_top = RESUMEN[RESUMEN.split == "test"].sort_values("ha consenso", ascending=False).iloc[0]
mostrar_escena(fila_top.crop_id, UBIC[fila_top.crop_id]["fechas"].index(fila_top.fecha), guardar=True)


## 3.9 Exportación
Todo queda en `/kaggle/working/salidas`:

| Archivo / carpeta | Contenido |
|---|---|
| `modulo1_indices/{crop_id}/{fecha}_indices.tif` | NDVI, GNDVI, NDRE, NDMI (int16, índice × 10000) |
| `modulo2_mascaras/{crop_id}/...` | por fecha: palma(t), válido, evaluable · por ubicación: palma de referencia, no-dosel, etiqueta, palma 2019 sin dosel |
| `modulo3_anomalias/{crop_id}/{fecha}_anomalias.tif` | 6 capas por detector + votos (255 = no evaluado) |
| `modulo3_anomalias/{crop_id}/{crop_id}_persistencia.tif` | fracción de fechas con anomalía de consenso |
| `resumen_anomalias_por_escena.csv`, `regiones_anomalas.csv` | tablas de resultados |
| `evaluacion_sintetica.csv`, `segmentacion_metricas.csv`, `modulo2_resumen.csv` | métricas |
| `modelos/` | U-Net, encoders SimCLR/BYOL, autoencoder, clásicos (`.joblib`), `configuracion.json` |

Con `COMPRIMIR_SALIDAS = True`, las carpetas de GeoTIFF se empaquetan en un `.zip` por módulo. Los GeoTIFF se
abren directamente en QGIS: están georreferenciados y las bandas tienen nombre.


In [ ]:
t0 = time.time()
for c in ids_inferencia:
    u = UBIC[c]
    for t, fecha in enumerate(u["fechas"]):
        escribir_tif(os.path.join(OUT_DIR, "modulo3_anomalias", c, f"{fecha}_anomalias.tif"),
                     u["anomalias"][t], u, "uint8", 255, NOMBRES_DET + ["votos"])
    escribir_tif(os.path.join(OUT_DIR, "modulo3_anomalias", c, f"{c}_persistencia.tif"),
                 np.nan_to_num(u["persistencia"], nan=-1).astype(np.float32), u, "float32", -1,
                 ["persistencia_consenso"])

configuracion = dict(
    bandas=BANDAS, indices=INDICES, canales=CANALES, media=MU.tolist(), desviacion=SD.tolist(),
    umbrales=UMBRALES, contamination=CONTAMINATION, consenso_min=CONSENSO_MIN, patch=PATCH,
    umbral_palma=UMBRAL_PALMA, frac_fechas_palma=FRAC_FECHAS_PALMA, ndvi_no_dosel=NDVI_NO_DOSEL,
    split={c: u["split"] for c, u in UBIC.items()},
    ssd={n: dict(medias=p.medias.cpu().tolist(), precisiones=p.precisiones.cpu().tolist()) for n, p in ssd.items()})
with open(os.path.join(MODELOS_DIR, "configuracion.json"), "w") as f:
    json.dump(configuracion, f)

if COMPRIMIR_SALIDAS:
    for sub in ("modulo1_indices", "modulo2_mascaras", "modulo3_anomalias"):
        carpeta = os.path.join(OUT_DIR, sub)
        if os.path.isdir(carpeta):
            shutil.make_archive(carpeta, "zip", carpeta)
            shutil.rmtree(carpeta)
print(f"Exportación lista en {time.time() - t0:.0f} s:")
for f in sorted(os.listdir(OUT_DIR)):
    ruta = os.path.join(OUT_DIR, f)
    tam = sum(os.path.getsize(os.path.join(r, x)) for r, _, fs in os.walk(ruta) for x in fs) if os.path.isdir(ruta) \
        else os.path.getsize(ruta)
    print(f"  {f:45s} {tam / 1e6:8.1f} MB")


## Cómo leer los resultados y limitaciones

**Qué se puede afirmar en la tesis:**
- **Módulo 2:** la U-Net reproduce GlobalOilPalm con el IoU/F1 de `segmentacion_metricas.csv` en ubicaciones no
  vistas, y entrega una máscara por fecha de 2025.
- **Módulo 3:** cada detector está calibrado para marcar ~3% de la palma normal. La AUROC con estrés sintético
  compara su **sensibilidad a cambios espectrales plausibles**. El acuerdo (Jaccard) y la persistencia muestran
  qué tan coherentes son entre sí y en el tiempo.

**Qué no se puede afirmar:** la precisión real de detección de enfermedades. Para eso hace falta una muestra de
verdad de campo; con unas decenas de puntos visitados se calcularía la precisión de verdad.

**Limitaciones conocidas:**
- **Etiquetas de 2019 frente a imágenes de 2025.** GlobalOilPalm es de 2019; las diferencias de la U-Net con él
  pueden ser errores o cambios reales en el terreno.
- **Sombras de nubes.** No siempre vienen en `cloud_mask.tif`. Una sombra no enmascarada aparece como anomalía
  aislada que no persiste.
- **Menos parches en pequeños productores.** Los lotes de pequeños productores (~60 ha) aportan pocos píxeles por
  escena. Conviene revisar sus resultados por separado.
- **Persistencia limitada.** Con 7–9 fechas por año, la persistencia es informativa pero gruesa.

**Para ajustar:**
- `CONTAMINATION` controla la sensibilidad global.
- `CONSENSO_MIN` controla la exigencia del consenso.
- `FRAC_FECHAS_PALMA` y `NDVI_NO_DOSEL` controlan la región de búsqueda.
- `TEMPERATURA_SIMCLR` (0.1–0.5) y `FUERZA_JITTER_ESPECTRAL` son los hiperparámetros más influyentes de SimCLR y
  BYOL.
